# Coronary Artery Disease Detection
## Data Exploration & Preprocessing Analysis

---

**Dataset:** CAD Cardiac MRI Dataset — [Kaggle](https://www.kaggle.com/datasets/danialsharifrazi/cad-cardiac-mri-dataset)  
**Classes:** Normal (0) · Sick/CAD (1)  
**Modality:** Cardiac MRI (JPEG slices)

---

### Notebook Contents
1. Setup & Configuration
2. Dataset Structure
3. Class Distribution
4. Sample Image Visualisation
5. Image Property Analysis
6. Pixel Intensity Analysis
7. Data Quality Check
8. Preprocessing Pipeline
9. Augmentation Preview
10. Summary

## 1 · Setup & Configuration

In [ ]:
import os
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from PIL import Image, ImageEnhance
from collections import defaultdict

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

# ── Styling ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

CLR_NORMAL = '#2a9d8f'
CLR_SICK   = '#e63946'
CLR_BG     = '#f8fafc'

# ── Dataset path ─────────────────────────────────────────
DATASET_PATH = r'C:\Users\yycha\Downloads\archive(3)'
CLASSES      = ['Normal', 'Sick']
IMG_SIZE     = 224

print('Dataset path:', DATASET_PATH)
print('Exists:', os.path.exists(DATASET_PATH))

## 2 · Dataset Structure

In [ ]:
def scan_dataset(dataset_path):
    """Walk dataset and collect all image paths with labels."""
    data = defaultdict(list)
    dir_counts = defaultdict(lambda: defaultdict(int))

    for label, cls in enumerate(CLASSES):
        class_dir = os.path.join(dataset_path, cls)
        for root, _, files in os.walk(class_dir):
            for f in files:
                if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                    path = os.path.join(root, f)
                    data[cls].append(path)
                    # Track which top-level directory this belongs to
                    rel = os.path.relpath(root, class_dir)
                    top_dir = rel.split(os.sep)[0]
                    dir_counts[cls][top_dir] += 1
    return data, dir_counts

print('Scanning dataset...')
data, dir_counts = scan_dataset(DATASET_PATH)

n_normal = len(data['Normal'])
n_sick   = len(data['Sick'])
n_total  = n_normal + n_sick

print(f'\n{"─"*40}')
print(f'  Normal images : {n_normal:>7,}')
print(f'  Sick images   : {n_sick:>7,}')
print(f'  Total images  : {n_total:>7,}')
print(f'  Normal dirs   : {len(dir_counts["Normal"]):>7}')
print(f'  Sick dirs     : {len(dir_counts["Sick"]):>7}')
print(f'{"─"*40}')

In [ ]:
# Directory-level image counts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Images per Patient Directory', fontsize=14, fontweight='bold', y=1.01)

for ax, cls, color in zip(axes, CLASSES, [CLR_NORMAL, CLR_SICK]):
    dirs  = sorted(dir_counts[cls].keys())
    counts = [dir_counts[cls][d] for d in dirs]
    bars = ax.barh(dirs, counts, color=color, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_title(f'{cls} — {len(dirs)} Patient Directories', color=color, fontweight='bold')
    ax.set_xlabel('Number of Images')
    for bar, count in zip(bars, counts):
        ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
                f'{count:,}', va='center', fontsize=8, color='#444')
    ax.set_facecolor(CLR_BG)

plt.tight_layout()
plt.savefig('static/eda_dir_counts.png', bbox_inches='tight')
plt.show()

## 3 · Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Distribution', fontsize=14, fontweight='bold')

# Bar chart
ax = axes[0]
bars = ax.bar(CLASSES, [n_normal, n_sick],
              color=[CLR_NORMAL, CLR_SICK], alpha=0.85,
              edgecolor='white', width=0.5)
for bar, count in zip(bars, [n_normal, n_sick]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{count:,}', ha='center', fontweight='bold', fontsize=12)
ax.set_ylabel('Number of Images')
ax.set_title('Image Count per Class')
ax.set_facecolor(CLR_BG)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart
ax = axes[1]
wedges, texts, autotexts = ax.pie(
    [n_normal, n_sick],
    labels=CLASSES,
    colors=[CLR_NORMAL, CLR_SICK],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 12}
)
for at in autotexts:
    at.set_fontweight('bold')
    at.set_color('white')
    at.set_fontsize(13)
ax.set_title('Class Split')

imbalance_ratio = n_normal / n_sick
print(f'Imbalance ratio  : {imbalance_ratio:.2f}:1 (Normal:Sick)')
print(f'Normal fraction  : {n_normal/n_total*100:.1f}%')
print(f'Sick fraction    : {n_sick/n_total*100:.1f}%')

plt.tight_layout()
plt.savefig('static/eda_class_dist.png', bbox_inches='tight')
plt.show()

## 4 · Sample Image Visualisation

In [ ]:
def load_gray(path):
    return np.array(Image.open(path).convert('L'))

# Sample 8 images from each class
samples_normal = random.sample(data['Normal'], 8)
samples_sick   = random.sample(data['Sick'],   8)

fig = plt.figure(figsize=(16, 9))
fig.patch.set_facecolor('#1a1a2e')
fig.suptitle('Sample Cardiac MRI Slices', color='white',
             fontsize=16, fontweight='bold', y=0.98)

for row, (samples, cls, color) in enumerate([
    (samples_normal, 'Normal', CLR_NORMAL),
    (samples_sick,   'Sick (CAD)', CLR_SICK)
]):
    for col, path in enumerate(samples):
        ax = fig.add_subplot(2, 8, row * 8 + col + 1)
        img = load_gray(path)
        ax.imshow(img, cmap='gray', vmin=0, vmax=255)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls, color=color, fontsize=11,
                          fontweight='bold', labelpad=4)
            ax.yaxis.set_label_position('left')
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(2)
            spine.set_visible(True)

plt.tight_layout(pad=0.3)
plt.savefig('static/eda_samples.png', bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

In [ ]:
# Side-by-side comparison: same position from Normal vs Sick
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Normal vs CAD — Side-by-Side Comparison',
             fontsize=14, fontweight='bold')

pairs_n = random.sample(data['Normal'], 5)
pairs_s = random.sample(data['Sick'],   5)

for col, (pn, ps) in enumerate(zip(pairs_n, pairs_s)):
    for row, (path, cls, color) in enumerate([
        (pn, 'Normal', CLR_NORMAL),
        (ps, 'Sick',   CLR_SICK)
    ]):
        ax = axes[row][col]
        ax.imshow(load_gray(path), cmap='gray')
        ax.axis('off')
        ax.set_facecolor('black')
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(2); spine.set_visible(True)
        if col == 0:
            ax.set_ylabel(cls, color=color, fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('static/eda_comparison.png', bbox_inches='tight')
plt.show()

## 5 · Image Property Analysis

In [ ]:
# Sample 500 images from each class for property analysis
SAMPLE_N = 500
sample_paths = {
    'Normal': random.sample(data['Normal'], SAMPLE_N),
    'Sick':   random.sample(data['Sick'],   SAMPLE_N),
}

widths  = defaultdict(list)
heights = defaultdict(list)
aspects = defaultdict(list)

print(f'Analysing {SAMPLE_N} images per class...')
for cls, paths in sample_paths.items():
    for p in paths:
        try:
            w, h = Image.open(p).size
            widths[cls].append(w)
            heights[cls].append(h)
            aspects[cls].append(w / h)
        except:
            pass

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Image Dimension Properties', fontsize=14, fontweight='bold')

for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    axes[0].hist(widths[cls],  bins=30, alpha=0.6, color=color, label=cls, edgecolor='white')
    axes[1].hist(heights[cls], bins=30, alpha=0.6, color=color, label=cls, edgecolor='white')
    axes[2].hist(aspects[cls], bins=30, alpha=0.6, color=color, label=cls, edgecolor='white')

axes[0].set_title('Width Distribution');  axes[0].set_xlabel('Pixels')
axes[1].set_title('Height Distribution'); axes[1].set_xlabel('Pixels')
axes[2].set_title('Aspect Ratio');        axes[2].set_xlabel('W / H')
for ax in axes:
    ax.legend(); ax.set_facecolor(CLR_BG)

plt.tight_layout()
plt.savefig('static/eda_dimensions.png', bbox_inches='tight')
plt.show()

for cls in CLASSES:
    print(f'{cls:8s}  W: {np.mean(widths[cls]):.0f}±{np.std(widths[cls]):.0f}  '
          f'H: {np.mean(heights[cls]):.0f}±{np.std(heights[cls]):.0f}  '
          f'AR: {np.mean(aspects[cls]):.3f}')

## 6 · Pixel Intensity Analysis

In [ ]:
# Compute mean pixel intensity per image (sample 300 per class)
SAMPLE_PIX = 300
intensities = defaultdict(list)
std_devs    = defaultdict(list)
mean_pixels = {'Normal': None, 'Sick': None}  # for histogram overlay

print('Computing pixel statistics...')
pix_samples = {'Normal': [], 'Sick': []}

for cls, paths in sample_paths.items():
    for p in paths[:SAMPLE_PIX]:
        try:
            arr = np.array(Image.open(p).convert('L'), dtype=np.float32)
            if arr.max() > 0:
                intensities[cls].append(arr.mean())
                std_devs[cls].append(arr.std())
                pix_samples[cls].append(arr.flatten())
        except:
            pass

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Pixel Intensity Analysis', fontsize=14, fontweight='bold')

# Mean intensity distribution
ax = axes[0]
for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    ax.hist(intensities[cls], bins=40, alpha=0.65, color=color, label=cls, edgecolor='white')
ax.set_title('Mean Pixel Intensity per Image')
ax.set_xlabel('Mean Intensity (0–255)')
ax.set_ylabel('Count')
ax.legend(); ax.set_facecolor(CLR_BG)

# Std deviation distribution
ax = axes[1]
for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    ax.hist(std_devs[cls], bins=40, alpha=0.65, color=color, label=cls, edgecolor='white')
ax.set_title('Pixel Std Dev per Image (Contrast)')
ax.set_xlabel('Std Deviation')
ax.legend(); ax.set_facecolor(CLR_BG)

# Aggregated pixel value histogram
ax = axes[2]
for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    flat = np.concatenate(pix_samples[cls][:50])  # 50 images flattened
    ax.hist(flat, bins=128, alpha=0.5, color=color, label=cls,
            density=True, edgecolor='none')
ax.set_title('Pixel Value Distribution (density)')
ax.set_xlabel('Pixel Value (0–255)')
ax.legend(); ax.set_facecolor(CLR_BG)

plt.tight_layout()
plt.savefig('static/eda_intensity.png', bbox_inches='tight')
plt.show()

print(f'\n{"Class":8s}  {"Mean Intensity":>15}  {"Mean Std Dev":>12}')
print('─' * 40)
for cls in CLASSES:
    print(f'{cls:8s}  {np.mean(intensities[cls]):>14.2f}  {np.mean(std_devs[cls]):>12.2f}')

In [ ]:
# Average image (mean of 50 samples per class)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Average MRI Appearance per Class', fontsize=14, fontweight='bold')

avg_imgs = {}
for cls in CLASSES:
    stack = []
    for p in sample_paths[cls][:50]:
        try:
            arr = np.array(Image.open(p).convert('L').resize((224, 224)), dtype=np.float32)
            if arr.max() > 0:
                stack.append(arr)
        except:
            pass
    avg_imgs[cls] = np.mean(stack, axis=0)

for ax, cls, color, title in zip(
    axes[:2], CLASSES, [CLR_NORMAL, CLR_SICK],
    ['Average Normal MRI', 'Average Sick (CAD) MRI']
):
    im = ax.imshow(avg_imgs[cls], cmap='gray')
    ax.set_title(title, color=color, fontweight='bold')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Difference map
diff = avg_imgs['Normal'].astype(float) - avg_imgs['Sick'].astype(float)
im = axes[2].imshow(diff, cmap='RdBu_r', vmin=-40, vmax=40)
axes[2].set_title('Difference (Normal − Sick)', fontweight='bold')
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('static/eda_avg_images.png', bbox_inches='tight')
plt.show()

## 7 · Data Quality Check

In [ ]:
print('Running data quality checks on sample...')
QUALITY_SAMPLE = 1000  # per class

quality = defaultdict(lambda: {'black': 0, 'corrupt': 0, 'tiny': 0, 'ok': 0})
black_examples = []

for cls, paths in sample_paths.items():
    for p in paths[:QUALITY_SAMPLE]:
        try:
            img = Image.open(p).convert('L')
            arr = np.array(img)
            w, h = img.size
            if arr.max() == 0:
                quality[cls]['black'] += 1
                if len(black_examples) < 4:
                    black_examples.append((p, cls))
            elif w < 32 or h < 32:
                quality[cls]['tiny'] += 1
            else:
                quality[cls]['ok'] += 1
        except Exception:
            quality[cls]['corrupt'] += 1

print(f'\n{"Issue":12}  {"Normal":>8}  {"Sick":>8}')
print('─' * 32)
for issue in ['black', 'corrupt', 'tiny', 'ok']:
    print(f'{issue:12}  {quality["Normal"][issue]:>8}  {quality["Sick"][issue]:>8}')

# Estimated black frame totals
est_black_n = int(quality['Normal']['black'] / QUALITY_SAMPLE * n_normal)
est_black_s = int(quality['Sick']['black']   / QUALITY_SAMPLE * n_sick)
print(f'\nEstimated black frames in full dataset:')
print(f'  Normal: ~{est_black_n:,}  |  Sick: ~{est_black_s:,}')

In [ ]:
# Show black frame examples if any found
if black_examples:
    fig, axes = plt.subplots(1, len(black_examples), figsize=(3 * len(black_examples), 3))
    fig.suptitle('Black Frame Examples (filtered out during training)',
                 fontsize=12, fontweight='bold', color=CLR_SICK)
    if len(black_examples) == 1:
        axes = [axes]
    for ax, (p, cls) in zip(axes, black_examples):
        ax.imshow(np.array(Image.open(p).convert('L')), cmap='gray', vmin=0, vmax=255)
        ax.set_title(cls, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig('static/eda_black_frames.png', bbox_inches='tight')
    plt.show()
else:
    print('No black frames found in this sample (may still exist in full dataset).')

## 8 · Preprocessing Pipeline

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def preprocess_steps(path):
    """Returns each preprocessing step as a numpy array."""
    # Step 1: Original (RGB)
    original = np.array(Image.open(path).convert('RGB'))
    # Step 2: Grayscale
    gray = np.array(Image.open(path).convert('L'))
    # Step 3: Resized grayscale
    resized = np.array(Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE)))
    # Step 4: Normalised [0,1]
    norm = resized.astype(np.float32) / 255.0
    # Step 5: 3-channel for EfficientNet
    rgb3 = np.stack([norm] * 3, axis=-1)
    # Step 6: ImageNet normalised (for display, clamp back to [0,1])
    imagenet_norm = (rgb3 - IMAGENET_MEAN) / IMAGENET_STD
    display_norm  = np.clip((imagenet_norm - imagenet_norm.min()) /
                            (imagenet_norm.max() - imagenet_norm.min()), 0, 1)
    return original, gray, resized, norm, rgb3, display_norm

sample_path = data['Normal'][42]
steps = preprocess_steps(sample_path)
step_labels = [
    'Original (RGB)',
    'Grayscale',
    f'Resized ({IMG_SIZE}×{IMG_SIZE})',
    'Normalised [0, 1]',
    '3-channel (for EfficientNet)',
    'ImageNet normalised',
]
cmaps = ['viridis', 'gray', 'gray', 'gray', 'gray', 'gray']

fig, axes = plt.subplots(1, 6, figsize=(18, 4))
fig.suptitle('Preprocessing Pipeline — Single Image Walkthrough',
             fontsize=13, fontweight='bold')

for ax, img, label, cmap in zip(axes, steps, step_labels, cmaps):
    if img.ndim == 2 or (img.ndim == 3 and img.shape[2] == 1):
        ax.imshow(img, cmap='gray', vmin=0, vmax=img.max() if img.max() > 1 else 1)
    else:
        ax.imshow(np.clip(img, 0, 1) if img.dtype == np.float32 else img)
    ax.set_title(label, fontsize=9, fontweight='bold')
    ax.axis('off')

# Add arrows between steps
for i in range(5):
    fig.text(0.135 + i * 0.152, 0.5, '→', fontsize=18,
             ha='center', va='center', color='#64748b',
             transform=fig.transFigure)

plt.tight_layout()
plt.savefig('static/eda_pipeline.png', bbox_inches='tight')
plt.show()

In [ ]:
# Pixel distribution: before vs after normalisation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Pixel Distribution: Before vs After Normalisation', fontsize=13, fontweight='bold')

for cls, color in zip(CLASSES, [CLR_NORMAL, CLR_SICK]):
    raw_pixels, norm_pixels = [], []
    for p in sample_paths[cls][:30]:
        try:
            arr = np.array(Image.open(p).convert('L'), dtype=np.float32)
            if arr.max() > 0:
                raw_pixels.extend(arr.flatten())
                norm_pixels.extend((arr / 255.0).flatten())
        except:
            pass
    axes[0].hist(raw_pixels,  bins=128, alpha=0.5, color=color, label=cls, density=True)
    axes[1].hist(norm_pixels, bins=128, alpha=0.5, color=color, label=cls, density=True)

axes[0].set_title('Raw (0–255)'); axes[0].set_xlabel('Pixel Value'); axes[0].legend()
axes[1].set_title('Normalised (0–1)'); axes[1].set_xlabel('Pixel Value'); axes[1].legend()
for ax in axes:
    ax.set_ylabel('Density'); ax.set_facecolor(CLR_BG)

plt.tight_layout()
plt.savefig('static/eda_norm_dist.png', bbox_inches='tight')
plt.show()

## 9 · Augmentation Preview

In [ ]:
def apply_augmentations(path):
    """Apply each augmentation individually for visualisation."""
    img = Image.open(path).convert('L').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img, dtype=np.float32) / 255.0

    augmented = {'Original': arr}

    # Horizontal flip
    augmented['H-Flip'] = arr[:, ::-1]

    # Vertical flip
    augmented['V-Flip'] = arr[::-1, :]

    # Brightness +10%
    aug = ImageEnhance.Brightness(img).enhance(1.2)
    augmented['Brightness ↑'] = np.array(aug, dtype=np.float32) / 255.0

    # Brightness -10%
    aug = ImageEnhance.Brightness(img).enhance(0.8)
    augmented['Brightness ↓'] = np.array(aug, dtype=np.float32) / 255.0

    # Contrast
    aug = ImageEnhance.Contrast(img).enhance(1.3)
    augmented['Contrast ↑'] = np.array(aug, dtype=np.float32) / 255.0

    return augmented

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
fig.suptitle('Data Augmentation Preview', fontsize=14, fontweight='bold')

for row, cls in enumerate(CLASSES):
    path = sample_paths[cls][7]
    augs = apply_augmentations(path)
    for col, (aug_name, aug_img) in enumerate(augs.items()):
        ax = axes[row][col]
        ax.imshow(aug_img, cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        color = CLR_NORMAL if cls == 'Normal' else CLR_SICK
        ax.set_title(aug_name, fontsize=9, fontweight='bold')
        for spine in ax.spines.values():
            spine.set_edgecolor(color); spine.set_linewidth(2); spine.set_visible(True)
        if col == 0:
            ax.set_ylabel(cls, color=color, fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('static/eda_augmentation.png', bbox_inches='tight')
plt.show()

## 10 · Summary

In [ ]:
summary = {
    'Total Images':           f'{n_total:,}',
    'Normal Images':          f'{n_normal:,} ({n_normal/n_total*100:.1f}%)',
    'Sick (CAD) Images':      f'{n_sick:,} ({n_sick/n_total*100:.1f}%)',
    'Class Imbalance Ratio':  f'{imbalance_ratio:.2f}:1 (Normal:Sick)',
    'Patient Directories':    f'Normal={len(dir_counts["Normal"])}, Sick={len(dir_counts["Sick"])}',
    'Image Format':           'JPEG (grayscale cardiac MRI slices)',
    'Black Frames (est.)':    f'Normal ~{est_black_n:,} | Sick ~{est_black_s:,}',
    'Input Size (model)':     f'{IMG_SIZE}×{IMG_SIZE}',
    'Normalisation':          '[0,1] then ImageNet mean/std',
    'Augmentations':          'H-flip, V-flip, Brightness ±10%, Contrast ±10%',
    'Train / Val / Test':     '80% / 10% / 10% (stratified)',
    'Class Weighting':        'Inverse frequency to handle imbalance',
}

print('='*55)
print('  DATASET & PREPROCESSING SUMMARY')
print('='*55)
for k, v in summary.items():
    print(f'  {k:<28}  {v}')
print('='*55)

In [ ]:
# Summary figure
fig = plt.figure(figsize=(14, 5))
fig.patch.set_facecolor('#1d3557')
ax = fig.add_subplot(111)
ax.set_facecolor('#1d3557')
ax.axis('off')

fig.text(0.5, 0.92, 'CAD Detection — EDA Summary',
         ha='center', fontsize=16, fontweight='bold', color='white')

stats = [
    ('Total Images', f'{n_total:,}', '📊'),
    ('Normal', f'{n_normal:,}\n({n_normal/n_total*100:.1f}%)', '✅'),
    ('Sick (CAD)', f'{n_sick:,}\n({n_sick/n_total*100:.1f}%)', '⚠️'),
    ('Imbalance', f'{imbalance_ratio:.2f}:1', '⚖️'),
    ('Input Size', f'{IMG_SIZE}×{IMG_SIZE}', '🖼️'),
    ('Modality', 'Grayscale\nCardiac MRI', '🫀'),
]

for i, (label, value, icon) in enumerate(stats):
    x = 0.08 + i * 0.155
    rect = mpatches.FancyBboxPatch((x - 0.06, 0.1), 0.12, 0.72,
                                    boxstyle='round,pad=0.02',
                                    linewidth=0, facecolor='#457b9d',
                                    transform=fig.transFigure, clip_on=False)
    fig.add_artist(rect)
    fig.text(x, 0.73, icon, ha='center', fontsize=18, transform=fig.transFigure)
    fig.text(x, 0.55, value, ha='center', fontsize=13, fontweight='bold',
             color='white', transform=fig.transFigure, va='top')
    fig.text(x, 0.18, label, ha='center', fontsize=9,
             color='#a8dadc', transform=fig.transFigure)

plt.savefig('static/eda_summary.png', bbox_inches='tight', facecolor='#1d3557')
plt.show()

print('\nAll EDA plots saved to static/')